In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import json
from pathlib import Path

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
ROOT_DIR = Path(dataset_config["path_processed"]) / "WOS" / "_CHAT"
SHARDS_DIR = ROOT_DIR
COMPLETED_DIR = ROOT_DIR / "completed"

# ------------------------------------------------------------
# 1) Build mapping: custom_id -> organization (from shard JSONL)
# ------------------------------------------------------------
custom_id_to_org = {}

for shard_path in sorted(SHARDS_DIR.glob("batch_*.jsonl")):
    # skip completed result files if they share prefix
    if shard_path.parent.name == "completed":
        continue

    print(f"[read shard] {shard_path.name}")
    with shard_path.open("r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            custom_id = obj.get("custom_id")
            body = obj.get("body", {})
            org = body.get("input")  # we wrote org here earlier

            if custom_id is not None and org is not None:
                custom_id_to_org[custom_id] = org

print(f"Loaded {len(custom_id_to_org):,} custom_id→organization mappings")

# ------------------------------------------------------------
# 2) Helper: extract model text from Responses API body
# ------------------------------------------------------------
def extract_text_from_body(body: dict) -> str:
    # Try convenience field (Azure/OpenAI variants)
    if isinstance(body.get("output_text"), str):
        return body["output_text"]

    output = body.get("output")
    if isinstance(output, list) and output:
        item = output[0]
        content = item.get("content")
        if isinstance(content, list) and content:
            part = content[0]
            # Most common layout: {"type": "output_text", "text": "..."}
            if isinstance(part.get("text"), str):
                return part["text"]
            # Some variants: {"text": {"value": "..."}}
            t = part.get("text")
            if isinstance(t, dict) and isinstance(t.get("value"), str):
                return t["value"]

    return ""

# ------------------------------------------------------------
# 3) Parse completed batch outputs: custom_id -> label
# ------------------------------------------------------------
rows = []

for out_path in sorted(COMPLETED_DIR.glob("*.jsonl")):
    print(f"[parse completed] {out_path.name}")
    with out_path.open("r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            custom_id = obj.get("custom_id")
            if custom_id is None:
                continue

            body = obj.get("response", {}).get("body", {})
            text = extract_text_from_body(body).strip()

            # Strict 0/1 rule
            is_us = 1 if text == "1" else 0

            org = custom_id_to_org.get(custom_id)
            rows.append(
                {
                    "custom_id": custom_id,
                    "organization": org,
                    "raw_text": text,
                    "is_us": is_us,
                }
            )

In [ ]:
df_results = pd.DataFrame(rows).drop_duplicates(subset=["custom_id"])[['organization', 'is_us']]
df_results

In [ ]:
df_filtered = df_results[df_results.is_us == 1]
df_filtered

In [ ]:
df_filtered[['organization']].to_csv(dataset_config['path_processed'] + 'WOS/POST00_US_firms_filtered.csv', index=False)